<a href="https://colab.research.google.com/github/HectorOik/UCL-Machine-Vision-Final-Lab/blob/main/Model_Inference_(FINAL_SUBMISSION_DO_NOT_TOUCH).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== INSTALL DEPENDENCIES =====
# !pip install boto3 -q
# !pip install opencv-python torch numpy torchvision tqdm
print("Setting up environment. May take a couple of minutes...")
!pip uninstall -y opencv-python opencv-python-headless -q
!pip install "opencv-python-headless<4.11" "mediapipe==0.10.14" boto3 -q
!pip install huggingface_hub

Setting up environment. May take a couple of minutes...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 137.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 w

In [ ]:
# Import the required libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os
import cv2
import numpy as np
from tqdm import tqdm
import time
import random
import mediapipe as mp
from sklearn.model_selection import train_test_split

# Please double, triple, quadruple check that the below code runs without errors before submitting.

## TODO 1 - Enter your HuggingFace username below:

In [ ]:
hf_username = "hector-oikonomidis"

## TODO 2 - Define your model EXACTLY as you did in your training code (otherwise there will be errors, and, possibly, tears).

Note below the classname is 'YourModelArchitecture'. That's because it literally needs to be YOUR MODEL ARCHITECTURE. This class definition is later referred to below in the 'load_model_from_hub' method. The architecture must match here, or it will not be able to instantiate the model weights correctly once it downloads them from HuggingFace. Pay very close attention to getting this right, please.

Replace the below code, and replace the corresponding line in the 'load_model_from_hub' method.

In [ ]:
# =============================================================================
# 1. MODEL DEFINITION (must match training)
# =============================================================================

class PoseRepNet(nn.Module):
  def __init__(self, input_dims=99, hidden_dims=64, num_classes=11):
    super().__init__()

    # ================
    #   Pose Encoder
    # ================

    # projects 99-dim pose -> 64-dim latent feature
    # applied to every frame independently
    self.embedding = nn.Sequential(
        nn.Linear(input_dims, 128),
        nn.ReLU(),
        nn.Linear(128, hidden_dims)
    )

    # ===================================
    #   Stripe Counter (3-Layer 2D CNN)
    # ===================================

    """Takes TxT similarity matrix as input"""
    self.cnn = nn.Sequential(
        # layer 1: 16 channels out
        nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.MaxPool2d(2), # 64x64 -> 32x32 dim

        # layer 2: 32 channels out
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2), # 32x32 -> 16x16 dim

        # layer 3: 64 channels out
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2), # 16x16 -> 8x8 dim

        # Safety layer: ensure output is 8x8
        # Doesn't do anything on correctly sized layer 3 output
        nn.AdaptiveAvgPool2d((8,8))
    )

    # ==============
    #   Classifier
    # ==============

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(8*8*64, 128), # 64 channels * 8x8 dim input
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, num_classes) # num_classes here is 11 since 0 is included but can be dynamically changed as class input
    )

  # cropping of similarity matrix is done outside this function
  def get_similarity_matrix(self, embeds):
    """
    Calculates self-similarity matrix.
    Input: (Batch, MAX_FRAMES, Hidden) -> Output: (Batch, 1, MAX_FRAMES, MAX_FRAMES)
    """
    embeds = F.normalize(embeds, p=2, dim=-1)
    sim = torch.bmm(embeds, embeds.transpose(1,2))
    return sim.unsqueeze(1) # add channel dim: (B, 1, MAX_FRAMES, MAX_FRAMES)

  def forward(self, x, actual_lengths):
      """
      x: (Batch, Max_frames, 99)
      actual_lengths: (Batch,) - True length of each video
      """
      batch_size = x.shape[0]

      # =======================
      #   Generate Embeddings
      # =======================
      # flatten batch & time for linear layer
      x_flat = x.view(-1, x.shape[-1])
      embeds_flat = self.embedding(x_flat)
      # reshape back to (B, T, 64)
      embeds = embeds_flat.view(batch_size, -1, embeds_flat.shape[-1])

      # =====================
      #   Similairty Matrix
      # =====================
      sim_matrix = self.get_similarity_matrix(embeds)

      # crop & resize (deal with variable length): makes matrix TxT
      processed_matrices = []

      for i in range(batch_size):
        length = int(actual_lengths[i].item())

        # safety check
        if length < 2: length = 2
        if length > sim_matrix.shape[2]: length = sim_matrix.shape[2]

        # remove 0 padding (matrix is now TxT)
        valid_slice = sim_matrix[i,:,:length,:length]

        # resize similarity matrix
        resized = F.interpolate(
            valid_slice.unsqueeze(0),
            size=(64,64), # target similarity matrix size
            mode='bilinear',
            align_corners=False
        )

        processed_matrices.append(resized)

      # stach back into a batch: (B, 1, 64, 64)
      cnn_input = torch.cat(processed_matrices, dim=0)

      # ================================
      #   2D Convolutions & Classifier
      # ================================

      features = self.cnn(cnn_input)
      logits = self.classifier(features)

      return logits


## Download the test data from s3, and create the corresponding dataset + dataloader.

There's no TODO for you here. This text is just here to explain to you what this code does.

In this instance, the test data IS the training data you were provided in the Model Training notebook. This is by design. You do not have access to the test data. This is a simple check to make sure the mechanics of this notebook work.

You should achieve the same accuracy here in this notebook, as you did in your previous notebook (random seed notwithstanding).

In [ ]:
# =============================================================================
# DOWNLOAD TEST DATA FROM S3
# =============================================================================

def download_test_data(bucket_name='training-and-validation-data',download_dir='./test-data'):
    s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

    bucket_name = 'prism-mvta'
    prefix = 'training-and-validation-data/'

    os.makedirs(download_dir, exist_ok=True)

    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    video_names = []

    for page in pages:
        if 'Contents' not in page:
            print("No files found at the specified path!")
            break

        print("Downloading test data:\n")
        for obj in tqdm(page['Contents']):
            key = obj['Key']
            filename = os.path.basename(key)

            if not filename:
                continue

            video_names.append(filename)
            local_path = os.path.join(download_dir, filename)
            # print(f"Downloading: {filename}")
            s3.download_file(bucket_name, key, local_path)

    print(f"\nDownloaded {len(video_names)} test videos")
    return download_dir


# ============================================================================= # DATASET AND DATALOADER =============================================================================

class Constants:
  MAX_FRAMES = 1000     # at 30 fps this is over 30 seconds, much longer than training videos. Nevertheless, 0 padding is cheap in this model framework
  POSE_DIMS = 33 * 3    # 33 landmarks * (x,y,z) = 99 dimensions (3D pose estimation)
  BATCH_SIZE = 64
  CACHE_DIR = './pose_cache' # save .npy files for ease of access during development
  VIDEO_DIR = './video-data'

# =================
#   Dataset Class
# =================

class PoseVideoDataset(Dataset): # used to be named VideoDataset
    """Dataset for loading videos from a folder. Labels from filename prefix."""
    def __init__(self, mode='cache', video_dir=Constants.VIDEO_DIR, cache_dir=Constants.CACHE_DIR, augment=False, max_frames=Constants.MAX_FRAMES):
        self.mode = mode
        self.cache_dir = cache_dir
        self.video_dir = video_dir
        self.max_frames = max_frames
        self.augment = augment
        self.augmenter = PoseAugmenter() if augment else None # PoseAugmenter not defined in this notebook as we don't use data augmentation logic (warning is fine)

        if self.mode == 'bootstrap':
          os.makedirs(self.cache_dir, exist_ok=True)
          self.video_files = sorted([f for f in os.listdir(video_dir) if f.endswith(('.mp4', '.avi', '.mov'))])
          # initialize mp pose only if we are actually processing raw videos
          self.mp_pose = mp.solutions.pose
          self.pose_model = self.mp_pose.Pose(
            static_image_mode=False,
            model_complexity=0, # 0=Lite (Fast)
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
          )
        else:
          self.video_files = sorted([f for f in os.listdir(cache_dir) if f.endswith('.npy')])

        # parse labels from file names
        self.labels = [
            int(f.split('_')[0]) for f in self.video_files
        ]


    def __len__(self):
        return len(self.video_files)

    def _normalize_landmarks(self, landmarks):
        """
        Input: list of 33 landmakrs objects
        Output: Normalized 99-dim vector (flattened)
        Logic: Center the hip at (0,0,0) and scale torso to unt length
        """
        # convert to numpy (33,3)
        coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])

        # center at hip: MediaPipe: 23=Left Hip, 24=Right Hip
        mid_hip = (coords[23] + coords[24]) / 2
        coords = coords - mid_hip # shift all points so hip is origin

        # scale by torso size: MediaPipe: 11=Left Shoulder, 12=Right Shoulder
        mid_shoulder = (coords[11] + coords[12]) / 2
        torso_size = np.linalg.norm(mid_shoulder - mid_hip) # euclidean distance

        if torso_size > 1e-6: coords = coords / torso_size

        # flatten and return
        return coords.flatten().astype(np.float32)

    def _process_video(self, video_path):
      """Extracts pose sequence from video. Returns (T, 99) array."""
      cap = cv2.VideoCapture(video_path)
      frames = []

      while cap.isOpened():
        ret, frame = cap.read()
        if not ret or len(frames) >= self.max_frames: break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.pose_model.process(frame_rgb)

        if results.pose_landmarks:
          landmarks = self._normalize_landmarks(results.pose_landmarks.landmark)
          frames.append(landmarks)
        else:
          frames.append(np.zeros(Constants.POSE_DIMS, dtype=np.float32))

      cap.release()

      if len(frames) < 0:
        return np.zeros((1, Constants.POSE_DIMS), dtype=np.float32)

      return np.array(frames)


    def __getitem__(self, idx):
      """
      Get video given idx
      Input: self, idx
      Output: padded pose video sequence, label, original video length
      """
      label = self.labels[idx]
      filename = self.video_files[idx]

      # during development used local cache to avoid constantly bootstrapping
      if self.mode == 'bootstrap':
        cache_path = os.path.join(self.cache_dir, filename + '.npy')
        if not os.path.exists(cache_path):
          pose_sequence = self._process_video(os.path.join(self.video_dir, filename))
          np.save(cache_path, pose_sequence)
        else:
          pose_sequence = np.load(cache_path)
      else: # cache mode: read npy directly (fast)
        try:
          pose_sequence = np.load(os.path.join(self.cache_dir, filename))
        except:
          # if cache is corrupted, re-process
          pose_sequence = np.zeros((self.max_frames, 99), dtype=np.float32)

      # augment datapoint if data augmentation is on
      if self.augment and self.augmenter:
        pose_sequence = self.augmenter.augment(pose_sequence)

      # =================
      #   Padding Logic
      # =================

      # get actual video length for cropping padding later
      actual_length = pose_sequence.shape[0]
      # copy actual data in padded buffer
      # if video was shorter, append zeros
      # if video was longer (handled in _process_video) fits perfectly
      limit = min(actual_length, self.max_frames)
      # pad video (for batching on GPU)
      padded_sequence = np.zeros((self.max_frames, Constants.POSE_DIMS), dtype=np.float32)
      padded_sequence[:limit] = pose_sequence[:limit]

      # return frames, label
      return (
          torch.tensor(padded_sequence),
          torch.tensor(label, dtype=torch.long),
          torch.tensor(limit, dtype=torch.long) # passing actual_length into model for later cropping
      )

# ========================
#   DataLoader Functions
# ========================

def get_dataloaders(video_dir=Constants.VIDEO_DIR, batch_size=Constants.BATCH_SIZE, val_split=0.2, expand_factor=5):
    """Create train and validation dataloaders."""

    # ======================================
    #   Split to Test and Validations Sets
    # ======================================

    base_dataset = PoseVideoDataset(mode='cache', augment=False)
    all_labels = [label for _, label, _ in base_dataset]

    train_indices, val_indices = train_test_split(
        list(range(len(base_dataset))),
        test_size=val_split,
        random_state=0,
        stratify=all_labels # such that all labels are present in the validation set
    )

    # data leakage as this loads all .npy files including synthetic data
    # this is intentional because some clases don't have data examples
    # however note that these classes are validating against synthetic data
    # and validation error results must be taken with a grain of salt
    val_dataset   = torch.utils.data.Subset(base_dataset, val_indices)

    train_subsets = [torch.utils.data.Subset(base_dataset, train_indices)]

    # =====================
    #   Data Augmentation
    # =====================
    for _ in range(expand_factor):
      train_subsets.append(torch.utils.data.Subset(
          PoseVideoDataset(mode='cache', augment=True),
          train_indices
      ))

    full_train_dataset = ConcatDataset(train_subsets)

    # ====================
    #   Weighted Sampler
    # ====================
    '''Deal with class imbalance - more relevant before augmentation but kept due to small dataset size'''
    val_size   = int(len(val_dataset))
    train_size = int(len(full_train_dataset))

    # get_dataloaders is called after bootstrap and synthetic data generation
    # pulling data from the cache and so sees synethetic data too
    train_labels = [base_dataset.labels[i] for i in train_indices]
    counts = np.bincount(train_labels, minlength=11)
    print(f'Train Class Counts: {counts}')

    weights = []
    for count in counts:
      weights.append(1.0 / count if count > 0 else 1.0) # don't rescale missing data weights

    weight_map = {label: weights[label] for label in range(len(weights))}

    base_sample_weights = [weight_map[label] for label in train_labels]
    full_sample_weights = base_sample_weights * (1 + expand_factor)

    sampler_weights = torch.DoubleTensor(full_sample_weights)
    sampler = WeightedRandomSampler(sampler_weights, num_samples=len(sampler_weights), replacement=True)

    # ==================
    #  Get DataLoaders
    # ==================

    train_loader = DataLoader(full_train_dataset, batch_size=batch_size, sampler=sampler)
    val_loader   = DataLoader(val_dataset, batch_size=val_size, shuffle=False)

    print(f"Train: {len(full_train_dataset)} videos, Val: {len(val_dataset)} videos")

    return train_loader, val_loader

## TODO 3 - Download your model from HuggingFace and instantiate it

Replace line 8 of the below code. Line 8 is where you instantiate YOUR MODEL ARCHITECTURE (which you re-defined above) with the weights you download from HuggingFace. Make sure you get the class name, and the arguments to the __init__ method correct.


This code just downloads the same model which you uploaded in the last notebook.

In [ ]:
# =============================================================================
# DOWNLOAD MODEL FROM HUGGING FACE
# =============================================================================

def load_model_from_hub(repo_id, num_classes=10):
    model_path = hf_hub_download(repo_id=repo_id, filename="model.pt")

    # hard coding 11 classes as my implementation includes 0
    model = PoseRepNet(input_dims=99, hidden_dims=64, num_classes=11)
    model.load_state_dict(torch.load(model_path, map_location='cpu'))

    print(f"Model loaded from {repo_id}")
    return model

model = load_model_from_hub(f"{hf_username}/mv-final-assignment",num_classes=10)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.pt:   0%|          | 0.00/2.29M [00:00<?, ?B/s]

Model loaded from hector-oikonomidis/mv-final-assignment


## TODO 4

Make sure the below code correctly evaluates your model performance on the given data!

This is your last chance to verify this before submission.

In [ ]:
def evaluate(model, test_loader, dataset, device):
    model.eval()
    correct = 0
    total = 0

    all_preds = []
    all_labels = []
    all_times = []

    print("\n")

    with torch.no_grad():
      # added lengths to successfully unpack dataloader
        for idx, (frames, labels, lengths) in enumerate(test_loader):
          # added lengths.to(device) (and lengths variable to save value)
            frames, labels, lengths = frames.to(device), labels.to(device), lengths.to(device)

            # Time the forward pass
            start_time = time.time()
            # passed lengths into model
            outputs = model(frames, lengths)
            if device.type == 'cuda':
                torch.cuda.synchronize()  # wait for GPU to finish
            end_time = time.time()

            inference_time = (end_time - start_time) * 1000  # ms
            all_times.append(inference_time)

            preds = outputs.argmax(dim=1)

            for i in range(labels.size(0)):
                batch_idx = idx * test_loader.batch_size + i
                video_name = dataset.video_files[batch_idx]
                pred = preds[i].item()
                true_label = labels[i].item()
                is_correct = "✓" if pred == true_label else "✗"

                print(f"{is_correct}  pred={pred}  true={true_label}  |  {inference_time:>7.1f}ms  |  {video_name}")

            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = correct / total
    return accuracy, all_preds, all_labels, all_times


# =============================================================================
# RUN INFERENCE
# =============================================================================

def run_inference(model, bucket_name='training-and-validation-data'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Download test data
    test_dir = download_test_data(bucket_name, './test-data')

    model = model.to(device)

    # Create dataloader
    # test_dataset = VideoDataset(test_dir, frame_size=(224, 224))
    # test_loader = DataLoader(
    #     test_dataset,
    #     batch_size=1,
    #     shuffle=False,
    #     num_workers=0,
    #     collate_fn=collate_fn
    # )
    #===============
    test_dataset = PoseVideoDataset(mode='bootstrap', video_dir=test_dir, augment=False, max_frames=Constants.MAX_FRAMES)
    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0
    )
    #===============

    print(f"\nRunning inference on {len(test_dataset)} test videos...")

    # Warmup (optional, helps get consistent GPU timings)
    if device.type == 'cuda':
        # dummy = torch.randn(1, 3, 1000, 224, 224).to(device)
        # replaced dummy initialization to follow shape requirements
        dummy_input   = torch.randn(1, Constants.MAX_FRAMES, Constants.POSE_DIMS).to(device)
        dummy_lengths = torch.tensor([Constants.MAX_FRAMES]).to(device)
        with torch.no_grad():
          # replaced model arguements based on dummy initialization
            _ = model(dummy_input, dummy_lengths)
        torch.cuda.synchronize()

    total_start = time.time()
    accuracy, preds, labels, times = evaluate(model, test_loader, test_dataset, device)
    total_end = time.time()

    # Summary
    num_correct = sum(p == l for p, l in zip(preds, labels))
    num_wrong = len(preds) - num_correct

    print("\n" + "="*50)
    print("SUMMARY")
    print("="*50)
    print(f"Total videos:         {len(preds)}")
    print(f"Correct:              {num_correct}")
    print(f"Incorrect:                {num_wrong}")
    print(f"")
    print(f"ACCURACY:             {accuracy*100:.2f}%")
    print(f"")
    print(f"Total time:           {total_end - total_start:.2f}s")
    print(f"Avg per video:        {sum(times) / len(times):.1f}ms")
    print(f"Min latency:          {min(times):.1f}ms")
    print(f"Max latency:          {max(times):.1f}ms")
    print("="*50)
    return accuracy, preds, labels

_, _, _ = run_inference(model)

Using device: cuda



100%|██████████| 77/77 [00:47<00:00,  1.61it/s]



Downloaded 77 test videos

Running inference on 77 test videos...




/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✓  pred=1  true=1  |      6.3ms  |  1_dksksjfwijf.mp4
✓  pred=2  true=2  |      6.3ms  |  2_dfsaeklnvvalkej.mp4
✓  pred=2  true=2  |      1.7ms  |  2_difficult_2.mp4
✓  pred=2  true=2  |      1.7ms  |  2_difficult_sdafkljsalkfj.mp4
✓  pred=2  true=2  |      2.1ms  |  2_dkdjwkndkfw.mp4
✓  pred=2  true=2  |      1.9ms  |  2_dkdmkejkeimdh.mp4
✓  pred=2  true=2  |      2.0ms  |  2_dkjd823kjf.mp4
✓  pred=2  true=2  |      1.8ms  |  2_dsalkfjalwkenlke.mp4
✓  pred=2  true=2  |      1.6ms  |  2_kling_20251205_Text_to_Video_On_a_sandy_4976_0.mp4
✓  pred=2  true=2  |      1.7ms  |  2_kling_20251206_Text_to_Video_Generate_a_71_1.mp4
✓  pred=2  true=2  |      1.8ms  |  2_sadfasjldkfjaseifj.mp4
✓  pred=2  true=2  |      1.7ms  |  2_sdafkjaslkclaksdjkas.mp4
✓  pred=2  true=2  |      1.7ms  |  2_sdfkjsaleijflaskdjf.mp4
✓  pred=2  true=2  |      1.7ms  |  2_sdjfhafsldkjhjk.mp4
✓  pred=2  true=2  |      1.6ms  |  2_sdkjdsflkjfwa.mp4
✓  pred=2  true=2  |      1.7ms  |  2_sdlfjlewlkjkj.mp4
✓  pred=2  tru